

<div style="max-width: 850px;">

# **LLM - The Transformer lifecycle**

---

This project is used to understand the lifecycle of transformers 
From data cleaning to speed up inference
We will keep it small such that we can train several transformers
All will be implemented from scratch
Later we can upgrade to use everything using hugging face interfaces


**Requirements**

These are requirements and how to install them in your environment

* `pip install datasets`

</div>



<div style="max-width: 850px;">

# The dataset

---

## Data curation and preparation

---

We will start small, using `tiny stories` dataset.
This dataset lives in huggingface. 

We will observe it's distribution and compute useful statistics from the dataset.

To process text we should also use a tokenizer. At the moment we are not interested in training one ourselves and we will use the `GPT2TokenizerFast` available in Huggingface. `GPT2TokenizeFast` implements a BPE (byte-pair encoding) tokenization.

</div>


In [ ]:
import matplotlib.pyplot as plt
import tqdm
import logging

import sys
from pathlib import Path

import numpy as np
import re
from wordcloud import WordCloud
from collections import Counter

import unicodedata
import html

from transformers import GPT2TokenizerFast

# To avoid getting blocked by sending many requests
from huggingface_hub import notebook_login

# imports from transformers lyfecycle

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import data_cycle

notebook_login()

logging.basicConfig(level=logging.INFO) 
_LOGGER = logging.getLogger(__name__)

data_path = '/Users/angelmartinez/src/transformers_lifecycle/datasets'



In [ ]:
# Load dataset
# Once downloaded them they will live in our environment
stories_train_dataset, stories_valid_dataset = data_cycle.load_tiny_stories_dataset()


In [ ]:
# display dataset information and a few examples
data_cycle.display_dataset_information(stories_train_dataset)
print(stories_train_dataset.select(range(10)).to_pandas())



<div style="max-width: 850px;">

## Dataset diagnosis
---

We run data diagnosis by computing and observing dataset statistics. These will help us to decide what pre-processing steps we need to apply to clean, standrize and filter documents. 

Usually this step is performed on a tokenized dataset. We will keep it simple and use `GPT2TokenizerFast`, which is a tokenizer trained using the BytePairing algorithm, and compute the following:

   - **Number of documents**: a document can be simply put as the basic unit in the dataset. This is a story within `tiny stories` dataset.
   - **Document length distribution**: the distribution help us identify documents that are too short or too long. 
   - **Vocabulary/Token distribution**: we will plot Zipf plots. These help us to identify if there are tokens that are too often repeating - such as html tags - and can indicate garbage or spurious dataset. For more context on these plots look [at this article.](https://www.nngroup.com/articles/zipf-curves-and-website-popularity/#:~:text=According%20to%20much%20available%20data%2C%20web%20use,number%20of%20elements%20that%20score%20very%20low)

**NOTE** `GPT2TokenizerFast` has a maximun length of 1024. Any document with longer length will be cliped by it.

</div>



In [ ]:
# Tokenizing dataset

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# apply a function to map and pre process the data to tokens
# for tokenizing the text and generate a sequence of tokens
def tokenize_preprocess_fn(row_example):
    return {
        'text': row_example['text'], 
        'tokenized': tokenizer(row_example['text'])['input_ids']
    }

stories_train_dataset = stories_train_dataset.map(
    tokenize_preprocess_fn,
    batched=True,
    batch_size=1000,
    num_proc=4,
    desc="Tokenizing text"
)


In [ ]:
# computing disrtibution of length and tokens

# vocabulary
vocabulary = tokenizer.get_vocab()

_LOGGER.info(f'Len of vocabulary: {len(vocabulary)}')
the_lengths = [
    len(dd['tokenized']) 
    for dd in tqdm.tqdm(stories_train_dataset, desc='Get lengths of data')
]

the_ids = []
for d in tqdm.tqdm(stories_train_dataset, desc='Gathering tokens'):
    the_ids.extend(d['tokenized'])

id_dist = {x:0 for x in set(the_ids)}
for k in tqdm.tqdm(the_ids, desc='Computing token distribution'):
    id_dist[k] += 1

id_dist_list = sorted([(k, v) for k, v in id_dist.items()], key=lambda x: x[0])


In [ ]:
# Looking to some parts of the dataset

# Let's look at the largest ocurrence. We will find that there is only a few
# tokens that are taking a lot of the counts -> over 10M occurences!
topk = 10
id_dist__ = sorted([(k, v) for (k, v) in id_dist_list], key=lambda x: x[1])

_LOGGER.info(f' The following are the {topk} tokens with largest ocurrence')
vocab_reversed = {v:k for k, v in vocabulary.items()}

for k, v in [[vocab_reversed[x[0]], x[1]] for x in id_dist__[-topk:]]:
    _LOGGER.info(f' {k} -> {v}')


In [ ]:
# plot counts and world cloud of tokens


mean, median, std = np.mean(the_lengths), np.median(the_lengths), np.std(the_lengths)

_LOGGER.info(f' Total of documents in the dataset: {len(stories_train_dataset)}')
_LOGGER.info(f' Document length statistics: mean:{mean}; median: {median}; std: {std}')
_LOGGER.info(f' Vocabulary size of tokenizer: {len(vocabulary)}')


fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0,0].hist(the_lengths, bins=50, color='slateblue', edgecolor='white')
axes[0,0].axvline(x=mean, linestyle='--', color='orange', label='mean')
axes[0,0].axvline(x=median, linestyle='--', color='red', label='median')

axes[0,0].axvline(x=mean+2*std, linestyle='--', color='green', label='mean+/- 2 std')
axes[0,0].axvline(x=mean-2*std, linestyle='--', color='green', label='mean+/- 2 std')

axes[0,0].legend()
axes[0,0].set_xlabel("Length of sequence (tokenized)")
axes[0,0].set_title('Histogram - document length')
axes[0,0].set_ylabel('Counts')
axes[0,0].grid(True, linestyle='--')

token_counts = data_cycle.ids_to_token_counts(id_dist, tokenizer)
max_words = 200

wc = WordCloud(
    width=1000,
    height=1000,
    background_color="white",
    max_words=max_words
)

wc.generate_from_frequencies(token_counts)
axes[0,1].imshow(wc)
axes[0,1].axis("off")
axes[0,1].set_title("Token Frequency (raw)")

counts = np.array(sorted(id_dist.values(), reverse=True))
ranks = np.arange(1, len(counts) + 1)
# ideal Zipf curve (scaled to match first point)
ideal = counts[0] / ranks

axes[1,0].loglog(ranks, counts, label='Empirical')
axes[1,0].loglog(ranks, ideal, linestyle="--", label="Ideal Zipf (1/r)")
axes[1,0].set_xlabel("Token rank (log)")
axes[1,0].set_ylabel("Token count (log)")
axes[1,0].set_title("Token Frequency Distribution / Zipf Plot")
axes[1,0].grid(True)
axes[1,0].legend()


display_counts = data_cycle.filter_token_counts_for_display(id_dist, tokenizer)
wc.generate_from_frequencies(display_counts)
axes[1,1].imshow(wc)
axes[1,1].axis("off")
axes[1,1].set_title("Token Frequency (no-punct)")



In [ ]:
# Observing the most and least ranked tokens
N = 10  # configurable

# Sort (token_id, count) by frequency
ranked_tokens = sorted(
    id_dist.items(),
    key=lambda x: x[1],
    reverse=True
)

def show_token(tokenizer, token_id):
    # Raw GPT-2 token representation + decoded human-readable text
    raw = tokenizer.convert_ids_to_tokens(token_id)
    decoded = tokenizer.decode([token_id])
    return raw, decoded


print(f"Top {N} highest-ranked tokens:")
for rank, (token_id, count) in enumerate(ranked_tokens[:N], start=1):
    raw, decoded = show_token(tokenizer, token_id)
    print(
        f"Rank {rank:>6} | "
        f"ID {token_id:>6} | "
        f"token={repr(raw):<18} | "
        f"text={repr(decoded):<18} | "
        f"count={count}"
    )

print(f"\nBottom {N} lowest-ranked tokens:")
total = len(ranked_tokens)

for rank, (token_id, count) in enumerate(
    ranked_tokens[-N:],
    start=total - N + 1
):
    raw, decoded = show_token(tokenizer, token_id)
    print(
        f"Rank {rank:>6} | "
        f"ID {token_id:>6} | "
        f"token={repr(raw):<18} | "
        f"text={repr(decoded):<18} | "
        f"count={count}"
    )


<div style="max-width: 850px;">

**Diagnosis**


* **Most frequent tokens**. These are punctuation tokens and lots of short length words. For training, keep punctuation and frequent tokens - **removing them breaks language modeling because the model must learn *syntax, spacing, sentence boundaries, formatting, etc.***. **It only becomes a problem** if the top tokens are things like `<, >, +, http, www, etc` or repeated boilerplate tokens from HTML, templates, logs, menus, tables, or corrupted text.

    * **Conclusion**. There are no unwanted tokens in this dataset. When removed punctuation and `articles` such as "the", "that" etc, most frequent tokens become pronouns like "he", "she" - which is normal in stories.


* **Length of documents have long tails**. We don't want documents on both tails - very short documents and very long documents. The length of the documents do not make them "be of bad quality", but it's more of a question of efficient training. For example, very short documents make training inefficient because **the model sees little context**. Short examples often include: titles, menu labels, broken web fragments, captions, etc. Very long documents will simply be cut by a `MAX_SEQUENCE_LENGTH` - loading them into memory will be inefficient. In practice document length will be kept in boundaries `MIN_SEQUENCE_LENGTH` and `MAX_SEQUENCE_LENGTH` with boundaries depending on the dataset.

    * **Conclusion**. There is a long tail for long documents, but overall the distribution looks good. During our exercise we will keep the length of ducuent short with `MIN_SEQUENCE_LENGTH=128` and `MAX_SEQUENCE_LENGTH=512`.


* **Zipf curves**. Zipf curves help us visualizaing the distributions of the tokens in the dataset by ranking: most frequent token has rank 1. This plot should be looking smooth and decreasing and spikes and clifts on the tail. We can diagnose the curve by visualizaing the ideal Zipf distribution `1/rank`.

    * **Conclusion**. The token-frequency distribution approximately follows a Zipf-like distribution, with token frequency decreasing roughly as a power law with rank. The highest-frequency tokens include punctuation, whitespace-related tokens, and common lexical/subword tokens, which explains the relatively flat head of the distribution. The middle portion of the distribution follows the 1/rank reference reasonably well. At high ranks, the empirical distribution falls below the ideal Zipf curve, forming a long tail of rare tokens. Inspection of these tokens shows meaningful low-frequency terms such as "Bigfoot" and "Iodine", suggesting that the tail contains legitimate story-specific vocabulary rather than simply noise. Therefore, rare tokens should not be removed solely based on their frequency.


</div>



<div style="max-width: 850px;">

## Pre-processing and data curation
---

**Data curation is the first and upmost component on quality control on LLMs**. Curation is performed as an iterative process where apply filters, models and heuristics to improve the overall quality. Common pre-processing steps are removing exact duplicates, removing semantic similar elements, and filter elements based on "explainability". Each step in the data curation cycle contributes in different ways to efficient learning of LLMs. 

Data curation follows a simple recipe:

1. **Normalise text**: unicode conversion, boilerplate, stripping, etc,
3. **Exact deduplication**: using hasing,
4. **Near deduplication**: using minhash and local sensitive hashing (LSH),
5. **Apply tokenization**, we can use the same BPE tokenizer,
6. **Filter by token length**: after tokenization we remove elements outside of `MIN_SEQUENCE_LENGTH` and `MAX_SEQUENCE_LENGTH`,
7. **Perplexity filtering**: using document "explainability" from a LLM as a meassure of the quality (can be a strong or a small LLM),
8. **[optional] Semantic deduplication**: converting documents to embeddings and using nearest neigbours,
9. **Iteratively curate** dataset by training and analysing outputs.


In the following we will implement each of these steps in our dataset.

</div>




<div style="max-width: 850px;">

### Step 1 - Normalizing text

---

In the normalization step we simply convert all the text to unicode, strip and remove more than 1 spaces. Specifically, we will focus in the following normalization

1. **Unicode normalization**. Normalize different unicode representations into a canonical form: curly quotes → standard quotes, accented characters → consistent encoding,
2. **Whitespace normalization**. Collapse multiple spaces, normalize tabs/newlines, strip leading/trailing spaces,
3. **Line break normalization**. Convert `\r\n`, `\r → \n` and collapse excessive newlines (e.g., >2 → 2),
4. **Control character removal**. Remove: non-printable characters, zero-width spaces and Unicode artifacts,
5. **Standardize quotes and punctuation**. “ ” → ", ‘ ’ → ', normalize dashes (— → -),
6. **Lowercasing reduces vocabulary entropy, but removes signal (e.g., named entities)** so it's better to keep casing.

We will apply these normalization steps on the hugging face dataset


</div>


In [ ]:

def normalize_text(text: str) -> str:
    text = normalize_unicode(text)
    text = clean_html(text)
    text = remove_control_chars(text)
    text = normalize_punctuation(text)
    text = normalize_whitespace(text)
    return text


def normalize_batch(batch):
    return {
        "text": [normalize_text(x) for x in batch["text"]]
    }


_LOGGER.info('Normalizing text in training data')
stories_train_dataset = stories_train_dataset.map(
    data_cycle.normalize_batch,
    batched=True,
    batch_size=1000,
    num_proc=4,   # adjust based on CPU cores
    desc="Normalizing text"
)

_LOGGER.info('Normalizing text in validation data')
stories_valid_dataset = stories_valid_dataset.map(
    data_cycle.normalize_batch,
    batched=True,
    batch_size=1000,
    num_proc=4,   # adjust based on CPU cores
    desc="Normalizing text"
)


In [ ]:
# Save to disk to later load
# train_dataset.save_to_disk(Path(data_path) / 'tinystories_train_dataset')
# valid_dataset.save_to_disk(Path(data_path) / 'tinystories_val_dataset')



<div style="max-width: 850px;">


### Step 2 -Exact deduplication with hashing

---

We will use a hash to remove exact duplicates. First, we will convert all the text to a hash - after normalization - using `hashlib.sha256`. This will create a set of hashes whifh to perform deduplication and make fast lookups.

</div>


In [ ]:
%load_ext autoreload
%autoreload 2

stories_train_dataset = stories_train_dataset.map(
    data_cycle.add_text_hash,
    batched=True,
    batch_size=1000,
    num_proc=4,
    desc="Train:Hashing normalized text",
)

stories_valid_dataset = stories_valid_dataset.map(
    data_cycle.add_text_hash,
    batched=True,
    batch_size=1000,
    num_proc=4,
    desc="Val:Hashing normalized text",
)


_LOGGER.info('Removing exact dedups in training dataset')
prev_n = len(stories_train_dataset)
stories_train_dataset = data_cycle.remove_exact_dedups_in_dataset(stories_train_dataset)
_LOGGER.info(f'Train: Removed: {(1-len(stories_train_dataset)/ prev_n)*100} % -> {len(stories_train_dataset)} vs {prev_n}')

_LOGGER.info('Removing exact dedups in validation dataset')
prev_n = len(stories_valid_dataset)
stories_valid_dataset = data_cycle.remove_exact_dedups_in_dataset(stories_valid_dataset)
_LOGGER.info(f'Val: Removed: {(1-len(stories_valid_dataset)/prev_n)*100} % -> {len(stories_valid_dataset)} vs {prev_n}')

                      


<div style="max-width: 850px;">


### Step 3 - Near deduplication

---

Finding near duplicates help us to remove documents in the dataset that are not exact duplicates based on text, but instead have similar structure and content. We can define that two documents are near duplicates if they contain similar local patterns in their `n-grams` that are usually constructed in word level. 

N-grams have the following properties:

1. Preserve local structure,
2. Are robust to small details,
3. Ignore global ordering,
4. Are sensitive to similar/copied structure.

Using the N-grams of two documents, the similarity becomes comparing how much overlap have the two sets:

$$
J(A,B) = \frac{|A\cap B|}{|A\cup B|}.
$$

The above is the *Jaccard similarity*. However, at scale this becomes infeasible given that the complexity is $O(N^2)$, $N$ equal to the length of both sets. The solution is to use **MinHashing**. MinHashing converts the documents to a signature and allows us to compute the above expression with the property:

$$
P(\text{MinHash}(A)=\text{MinHash}(B)) = J(A,B).
$$

**The intuition** behind is that, for a permutation of an n-gram that is in $A\cup B$ it must be in: 1) in $A\cap B$, or 2) must be in A or B. The probability that lies in the intersection is $\frac{|A\cap B|}{|A\cup B|}$.

To implement **MinHashing** we simulate permutations with hash functions $h_i$ to create a document signature:

$$
m_i = h_i(g), g\in A.
$$

Then

$$
\text{signature}(A) = [m_1, m_2, m_3, ..., m_K],
$$

for $k=K$ hash functions. Then we compare two documents by computing:

$$
\hat{J}(A, B) = \frac{1}{K} \sum_{i=1}^K I[m_i(A)=m_i(B)].
$$

However, this computation is still $O(N^2)$. To speed up computation we turn our eyes to LSH (local sensitive hashing) to **help us to only compute the expensive comparison metric on documents that are similar in some bands**. Specifically, LSH performs the following steps:

1. Split each MinHash signature into $b$ bands, each containing $r$ rows:

$$
n = b\cdot r,
$$

where \(n\) is the signature length.

2. Hash each band independently into a hash table/bucket.

3. Define candidate pairs: two documents $A,B$ become a candidate pair if their signatures are identical in all $r$ rows of at least one band.

4 Only for these candidate pairs, compute a more precise similarity measure, such as $\hat{J}(A, B)$.


For example, for a two component band, a signature can become:
   
$$
\text{signature}(A) = [[m_1, m_2], [m_3, m_4], ..., [m_{K-1}, m_K]].
$$

Luckily, there are packages in Python that can help us computing these permutations - MinHashes and similarities with LSH. In this exercise we will use `datasketch`. In practice, we need to tune the following parameters:

- **`NGRAM size` — controls the representation.** We split each document into overlapping $n$-grams before computing MinHash. Smaller $n$-grams create more overlap between documents and are more tolerant of small textual changes, but can make unrelated documents appear similar. Larger $n$-grams capture more specific context and reduce accidental matches, but become more sensitive to small edits. **Trade-off:** smaller $n$ → higher recall / lower precision; larger $n$ → higher precision / lower recall.

- **`Number of permutations` — controls the accuracy of MinHash.** This determines the number of hash functions used to construct each MinHash signature. More permutations give a more accurate estimate of the true Jaccard similarity and reduce the variance of $\hat J(A,B)$, but increase hashing cost, memory usage, and LSH index size. **Trade-off:** more permutations → better similarity estimates but more compute and memory; fewer → faster and cheaper but noisier estimates.

- **`Similitud threshold` — controls which similarities LSH targets.** This tells LSH approximately where we want the transition between unlikely and likely candidate pairs to occur. A higher threshold retrieves fewer, more strongly similar candidates but risks missing useful moderately similar pairs. A lower threshold generates more candidates, improving recall but increasing false positives and downstream comparisons. **Trade-off:** higher threshold → higher precision / lower recall; lower threshold → higher recall / lower precision.

- **\(b\) and \(r\) — control the LSH banding behavior.** The MinHash signature is divided into \(b\) bands containing $r$ rows each, roughly satisfying $br=\text{NUM\_PERM}$. Two documents become candidates when they match on all \(r\) rows of at least one band. More bands with fewer rows make LSH more permissive; fewer bands with more rows make it stricter. **Trade-off:** more bands → higher recall / more false positives; more rows per band → higher precision / more false negatives.


</div>


In [ ]:
%load_ext autoreload
%autoreload 2
    
_LOGGER.info('Removing LSH dedups in training dataset')
train_keep, train_duplicates = data_cycle.get_unique_by_lsh(
    stories_train_dataset, num_proc=16, batch_size=256
)
prev_n = len(stories_train_dataset)
stories_train_dataset = stories_train_dataset.select(train_keep)

In [ ]:
_LOGGER.info(f'Train: Removed: {(1-len(stories_train_dataset)/ prev_n)*100} % -> {len(stories_train_dataset)} vs {prev_n}')


In [ ]:

_LOGGER.info('Removing LSH dedups in validation dataset')
prev_n = len(stories_valid_dataset)
val_keep, val_duplicates = data_cycle.get_unique_by_lsh(
    stories_valid_dataset, num_proc=16, batch_size=256
)
stories_valid_dataset = stories_valid_dataset.select(val_keep)

_LOGGER.info(f'Val: Removed: {(1-len(stories_valid_dataset)/ prev_n)*100} % -> {len(stories_valid_dataset)} vs {prev_n}')




<div style="max-width: 850px;">

### Step 4 - Tokenization

---

A dataset consist of documents. However, dataset is usually regarded as amount of tokens it cointains. Even more, when training a model we usually refer to the total count of tokens the model saw during training considering epochs too - e.g. GPT4 was trained on XX tokens. GPT models use a **Byte-Pair Encoding (BPE)** tokenizers, which decompose words into sub-words based on the most frequent pair of characters in subwords.

As an example, consider the input dataset `'low', 'lower', 'lowest'`. BPE works as follows

1. **Initialize vocabulary** - start with all unique characters as tokens including a token for unknown:

Vocabulary: `'l', 'o', 'w', 'e', 'r', 's', 't', '<unk>'`

2. **Count token pairs in the dataset**

```
(l,o) = 3    (w,e) = 2   (e,s) = 1
(o,w) = 3   (e,r) = 2   (s,t) = 1
```

3. **Merge most frequent pair into new subword token**:

Vocabulary after merging `(l,o)`: `'lo', 'w', 'e', 'r', 's', 't', '<unk>'`

4. **Repeat the process until desired granularity**:

Final vocabulary : `'low', 'lower', 'est', '<unk>'`

5. **The dictionary of vocabulary comprise of**:

`{'low':0, 'lower':1, 'est':3, '<unk>':3}`


We will apply a BPE tokenizer to our training and validation data. Specifically, we will use the tokenizer used for GPT2 models. Later we will compare the statisctics from the clean data w.r.t. the raw dataset.

</div>


In [ ]:

def preprocess_tokenizer_update(batch):
    return {'tokenized': tokenizer(batch['text'])['input_ids']}


_LOGGER.info('Tokenizing training dataset')
stories_train_dataset = stories_train_dataset.map(
    preprocess_tokenizer_update,
    batched=True,
    batch_size=512,
    num_proc=16,
    desc='Tokenizing train data'
)

_LOGGER.info('Tokenizing validation dataset')
stories_train_dataset = stories_train_dataset.map(
    preprocess_tokenizer_update,
    batched=True,
    batch_size=512,
    num_proc=16,
    desc='Tokenizing val data'
)




<div style="max-width: 850px;">


### 5 Perplexity computation

---

In hard terms, perplexity measures the spread (or uncertainty) of a distribution. For a discrete distribution:

$$
PPL(x) = \exp \left( - \sum_x P(x) \log P(x) \right) .
$$

For data curation, Perplexity helps removing examples that are likely outliers in the data distribution, i.e. it scores low under a distribution. In language modeling we use a language model $P_{\theta}$ as the distribution and compute the cross entropy of a sequence $x_{1:T}$ and its one-hot labels $y_{1:T}$:

$$
l(x) =   - \sum_{t,k} y_{tk} \log P_{\theta}(k | x_{<t}).
$$

and compute the perplexity using the previous formulation. Another formulation can be to compute the **mean perplexity per token** (instead of by sequence) averaging over the total number of tokens in the sequence $PPL(x) = \exp (l(x)/T)$ . 

Perplexity values can be interpreted as follows:

* **Low PPL**: the sequence is predictable under the reference model $P_{\theta}$. This means the sequence is fluent, in-domain, and it's a good signal of **good data**.
* **High PPL**: The text is an outlier. We can interpert it as noise, corruption, wrong language, etc, and it's signal of **noise data**.

**Note that PPL asumes a good reference LM $P_{\theta}$** and $PPL$ is thighted to the data distribution $P_{\theta}$ was trained with.


#### Statregies to use perplexity 

Most strategies rely on thresholding, either a hard threshold or **using a Quantile from the distribution**, e.g. remove 5-20% of the largest values. When the dataset consist of different domains, e.g. code, prose, books, etc, the best approach is to perform perplexity filtering by domain:

1. Cluster data by domain,
2. Filter data by PPL using the quantiles found by domain.


**To diagnose a dataset** before filtering by PPL we should look at

1. Histogram of per-document vs NNL/PPL,
2. PPL vs token length,
3. PPL by domain source,
4. Look at the examples from the tails.


In this exercise we will use GPT2 model to compute the perplexity and apply some filtering.


</div>


In [ ]:
# methods for computing perplexity
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel


@torch.no_grad()
def compute_sequece_perplexity_window(seq_tokens, model, device):
    # seq_tokens is a lisst of token ids -> to pytorch_tensor
    # convert to batch shape from shepe (N_TOKENS) to (1, N_TOKENS)
    input_tokens = torch.LongTensor(seq_tokens).unsqueeze(0).to(device)
    sequence_len = len(seq_tokens)

    max_chunk_length = 1024
    stride = 512
    lln_sum = 0
    total_tokens = 0

    # if sequence is longer thatn expected chunk_length we stride with window
    for begin in range(0, sequence_len, stride):
        end = min(begin+max_chunk_length, sequence_len)

        # compute the logits and with shape (B, sequence_length, vocab_size)
        input_tokens_ = input_tokens[:, begin:end]
        logits = model(input_tokens_).logits
        vocab_size = logits.size(-1)
        window_len = end-begin

        # GPT predicts tokens given x_t starts from x_{t+1} until T+1.
        # input sequence: [x_1, x_2, ..., x_T]
        # predicted sequence: [x_2, ..., x_{T+1}]
        # We cut the sequences to have elements in range 1:T
        logits_shifted = logits[:, :-1, :] # -> from 2 to T
        tgt = input_tokens_[:, 1:] # from 2 to T

        # We need to avoid double counting due to the nature of overlapping windows
        # and strinding, so we will get a mask and remove elements that overlapped
        # with the previous window -> length of window minus the stride
        n_overlap_tokens = window_len - stride
        the_logits = logits_shifted[:, n_overlap_tokens:, :]
        the_tgt = tgt[:, n_overlap_tokens:]

        # now we compute the negative log likelihood but do not average given that
        # perplexity will be computed per-token
        nll = F.cross_entropy(
            the_logits.view(-1, vocab_size), 
            the_tgt.view(-1), 
            reduction='none'
        )

        lln_sum += nll.sum().item()
        total_tokens += the_logits.size(1)

    if total_tokens == 0:
        return {'ppl': float("inf"), 'nll': float("inf")}

    nll_mean = lln_sum / total_tokens
    ppl = torch.exp(torch.tensor(nll_mean)).item()
    return {
        'ppl': ppl,
        'nll': nll
    }

@torch.no_grad()
def compute_batch_perplexity(batch, model, device):
    sequences = batch['tokenized']

    # pad sequences to the longest in the batch
    inputs = tokenizer.pad(
        {'input_ids': sequences},
        padding=True,
        return_tensors='pt'
    ).to(device)

    # attention mask -> 1 for all good entries, 0 for pad entries
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    ).logits

    # shift predictions and targets
    # input sequence: [x_1, x_2, ..., x_T]
    # predicted sequence: [x_2, ..., x_{T+1}]
    # We cut the sequences to have elements in range 1:T
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    shift_mask = attention_mask[:, 1:].contiguous()

    # compute per-token NLL
    token_nll = F.cross_entropy(
        shift_logits.reshape(-1, logits.size(-1)),
        shift_labels.reshape(-1),
        reduction='none'
    ).reshape(shift_labels.shape)

    # ignore padding tokens
    token_nll = token_nll * shift_mask

    # Average NLL for each sequence
    token_counts = shift_mask.sum(dim=1)
    mean_nll = token_nll.sum(dim=1) / token_counts.clamp(min=1)
    ppl = torch.exp(mean_nll)

    # Mark sequences with no predicted tokens as undefined
    # handling sequences with less than 2 tokens
    invalid = token_counts == 0
    mean_nll[invalid] = float('nan')
    ppl[invalid] = float('nan')

    return {
        'ppl': ppl.cpu().tolist(),
        'nll': mean_nll.cpu().tolist()
    }
    

@torch.no_grad()
def compute_sequece_perplexity(seq_tokens, model, device):
    # seq_tokens is a lisst of token ids -> to pytorch_tensor
    # convert to batch shape from shepe (N_TOKENS) to (1, N_TOKENS)
    input_tokens = torch.LongTensor(seq_tokens).unsqueeze(0).to(device)
    sequence_len = len(seq_tokens)

    logits = model(input_tokens).logits
    vocab_size = logits.size(-1)

    # GPT predicts tokens given x_t starts from x_{t+1} until T+1.
    # input sequence: [x_1, x_2, ..., x_T]
    # predicted sequence: [x_2, ..., x_{T+1}]
    # We cut the sequences to have elements in range 1:T
    logits_shifted = logits[:, :-1, :] # -> from 1 to T-1
    tgt = input_tokens[:, 1:] # from 2 to T

    # now we compute the negative log likelihood but do not average given that
    # perplexity will be computed per-token
    nll = F.cross_entropy(
        logits_shifted.view(-1, vocab_size), 
        tgt.view(-1), 
        reduction='none'
    )

    nll = nll.mean()
    ppl = torch.exp(nll)
    return {
        'ppl': ppl.item(),
        'nll': nll.item()
    }

def compute_token_len_batch(document_batch):
    return {
        'token_len': [len(x) for x in document_batch['tokenized']] 
    }



In [ ]:
# Filter by length

train_dataset_unique_lsh = train_dataset_unique_lsh.map(
    compute_token_len_batch,
    batched=True,
    batch_size=1000,
    num_proc=4,
    desc="Computing token lengths: train set",
)

val_dataset_unique_lsh = val_dataset_unique_lsh.map(
    compute_token_len_batch,
    batched=True,
    batch_size=1000,
    num_proc=4,
    desc="Computing token lengths: val set",
)


# First we will remove docuements outside of a range of tokens
MIN_TOKENS = 32
MAX_TOKENS = 1024

train_dataset_unique_lsh_inrange = train_dataset_unique_lsh.filter(
    lambda x: MIN_TOKENS <= x["token_len"] <= MAX_TOKENS,
    num_proc=4,
    desc="Filtering by token length: train dataset",
)

val_dataset_unique_lsh_inrange = val_dataset_unique_lsh.filter(
    lambda x: MIN_TOKENS <= x["token_len"] <= MAX_TOKENS,
    num_proc=4,
    desc="Filtering by token length: val dataset",
)

_LOGGER.info(f'After filtering train -> old: {len(train_dataset_unique_lsh)} and new: {len(train_dataset_unique_lsh_inrange)}')
_LOGGER.info(f'After filtering val -> old: {len(val_dataset_unique_lsh)} and new: {len(val_dataset_unique_lsh_inrange)}')




In [ ]:
# Use different models here

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
model.eval()


_LOGGER.info('Computing perplexity for training set')
train_dataset_unique_ppl = train_dataset_unique_lsh_inrange.map(
    compute_batch_perplexity,
    batched=True,
    batch_size=512,
    num_proc=16,
    fn_kwargs={'model': model, 'device': device}
)
#print(val_dataset_unique_lsh[0].keys())

_LOGGER.info('Computing perplexity for validation set')
val_dataset_unique_ppl = val_dataset_unique_lsh_inrange.map(
    compute_batch_perplexity,
    batched=True,
    batch_size=512,
    num_proc=16,
    fn_kwargs={'model':model, 'device':device}
)

# _LOGGER.info(f'{train_dataset_unique_lsh[0].keys()}')

In [ ]:

ppl, nll, length, texts, token_ids = [], [], [], [], []

for x in tqdm.tqdm(train_dataset_unique_ppl, total=len(train_dataset_unique_ppl)):
    ppl.append(x['ppl'])
    nll.append(x['nll'])
    length.append(x['token_len'])
    texts.append(x['text'])
    token_ids.extend(x['tokenized'])


In [ ]:
N = len(ppl)
ppl_sorted = sorted([(x,y) for x,y in zip(ppl, texts)], key=lambda x:x[0])
qq_85 = ppl_sorted[int(N*0.85)][0]
qq_95 = ppl_sorted[int(N*0.95)][0]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].hist(ppl, bins=50, color='slateblue', edgecolor='white')
axes[0].axvline(x=qq_85, linestyle='--', color='green', label='Q85')
axes[0].axvline(x=qq_95, linestyle='--', color='orange', label='Q95')

axes[0].legend()
axes[0].set_xlabel("Perplexity")
axes[0].set_title('Histogram - Perplexity under GPT2')
axes[0].set_ylabel('Counts')
axes[0].grid(True, linestyle='--')


axes[1].scatter(ppl, length, color='slateblue', alpha=0.1)

axes[1].set_xlabel("Perplexity")
axes[1].set_title('Perplexity vs Doc Token Length')
axes[1].set_ylabel('Doc Length')
axes[1].grid(True, linestyle='--')


In [ ]:


_LOGGER.info('Document with largest PPL:')
_LOGGER.info(ppl_sorted[-1][1])

Q95 = int(N*0.95)
_LOGGER.info('Set of 10 documents with larger PPL > Q95:')
_LOGGER.info(ppl_sorted[Q95:(Q95+10)])


In [ ]:
# We will remove elements in the dataset that are larger than the Quantil-95 of the distribution
# Basically we will remove the largest 5%


train_dataset_final = train_dataset_unique_ppl.filter(
    lambda x: x['ppl'] <= qq_85,
    num_proc=4,
    desc="Filtering by ppl <= Q85: train dataset",
)

val_dataset_final = val_dataset_unique_ppl.filter(
    lambda x: x['ppl'] <= qq_85,
    num_proc=4,
    desc="Filtering by ppl <= Q85: val dataset",
)



In [ ]:
_LOGGER.info(f'Size of training set: {len(train_dataset_final)}')
_LOGGER.info(f'Size of val set: {len(val_dataset_final)}')

token_ids_final = []
for x in tqdm.tqdm(train_dataset_final, total=len(train_dataset_final)):
    token_ids_final.extend(x['tokenized'])

_LOGGER.info('Computing token distribution')
id_dist_final = {x:0 for x in set(token_ids_final)}
for k in tqdm.tqdm(token_ids_final):
    id_dist_final[k] += 1

In [ ]:


fig, axes = plt.subplots(1, 2, figsize=(12, 6))


counts_f = np.array(sorted(id_dist_final.values(), reverse=True))
ranks_f = np.arange(1, len(counts_f) + 1)
# ideal Zipf curve (scaled to match first point)
ideal_f = counts_f[0] / ranks_f

axes[0].loglog(ranks_f, counts_f, label='Empirical Final')
axes[0].loglog(ranks, counts, label='Empirical Raw', color='purple')
axes[0].loglog(ranks_f, ideal_f, linestyle="--", label="Ideal Zipf (1/r)")
axes[0].set_xlabel("Token rank (log)")
axes[0].set_ylabel("Token count (log)")
axes[0].set_title("Token Frequency Distribution / Zipf Plot")
axes[0].grid(True)
axes[0].legend()

display_counts_f = filter_token_counts_for_display(id_dist_final, tokenizer)
wc.generate_from_frequencies(display_counts_f)
axes[1].imshow(wc)
axes[1].axis("off")
axes[1].set_title("Token Frequency (no-punct)")




<div style="max-width: 850px;">


## Different dataset domains
---

We will also experiment with another dataset to include it in the mix and experiment with different cleaning strategies


</div>


In [ ]:


code_ds = load_dataset(
    "code_search_net",
    "python",
    split="train",
    streaming=True,
)

# Small experimental sample
code_ds = code_ds.take(20000)



## 2 Transformer architecture

In these experiments we will use **GPT-like** architecture that uses a Decoder-only architecture and we will implement it from scratch using only the attention blocks from Pytorch. Contrary to the original Transformer encoder-decoder architecture from **Attention is all you Need** GPT architectures rely on a Decoder-only architecture, removing the cross-attention block between queries and encoder memory. Additionally, the architecture uses the following modifications: 1) a pre-normalization mechanics, 2) a dropout at the token embeddings and after the masked attention, and 3) an MLP as the `pointwise layer` with GELU activations.

**Where is the memory in Decoder-only architectures?** The memory appears implicit in the context of the input sequence. The longer the sequence become, the more context available - hence the memory - to predict the next token in the sequence. In causal attention tokens in position $t$ can attent to tokens in position $<t$ and use them as memory.

**The architecture of the Transformer decoder** We can see it as the following diagram:

```
    INPUT TOKENS
        ↓
    Embedding + Positional Encoding
        ↓
    Dropout
        ↓
 -------------------------------
|      Norm                     |
|       ↓                       |
|   Masked Self-Attention       |
|       ↓                       |
|   Dropout                     |
|       ↓                       |
|   Add + Norm                  |
|       ↓                       |
|   Feed Forward (MLP)          |  xN
|       ↓                       |
|      Add                      |
 -------------------------------
        ↓
       Norm
        ↓
   Linear + Softmax
```



In [ ]:
import torch
import torch.nn as nn


class GPTDecoderLayer(nn.Module):
    def __init__(self,
                 model_dim=256,
                 dim_ffn=2048,
                 n_heads=8,
                dropout=0.1):
        super(GPTDecoderLayer, self).__init__()
        self._model_dim = model_dim
        self._dim_ffn = dim_ffn
        self._n_heads= n_heads
        self._dropout = dropout

        # Self attention block
        self._self_attn = nn.MultiheadAttention(
            model_dim, n_heads, dropout=dropout
        )

        self._mlp = nn.Sequential(
            nn.Linear(model_dim, dim_ffn),
            nn.GELU(),
            nn.Linear(dim_ffn, model_dim),
            nn.Dropout(dropout),
        )
    
        self._norm1 = nn.LayerNorm(model_dim)
        self._norm2 = nn.LayerNorm(model_dim)

        self._dropout1 = nn.Dropout(dropout)

    def forward_pre(self, input_context, key_padding_mask=None):
        """
        Args:
            input_context: shape [sequence_length, batch_size, model_dim]
            key_padding_mask: shape [sequence_length, batch_Size]
        """
        seq_length = input_context.size(0)
        device = input_context.device
        _LOGGER.info(f'Size of input layer: {input_context.size()}')

        # input shape: (seq_len, batch_size, model_dim) -> output shape
        output = self._norm1(input_context)
        _LOGGER.info(f'Size of norm layer: {output.size()}')

        # construct matrix with upper triangle with -inf
        # Create first a matrix of booleans with 1 on the uper triangle and
        mask_look_ahead = ~torch.tril(torch.ones(seq_length, seq_length, dtype=torch.bool))
        mask_look_ahead = mask_look_ahead.float().masked_fill(mask_look_ahead, float("-inf")).to(device)
        # print(mask_look_ahead)

        # output_shape: (seq_len, batch_size, model_dim)
        attn, attn_weights = self._self_attn(
            query=output, key=output, value=output, 
            attn_mask=mask_look_ahead, 
            key_padding_mask=key_padding_mask
        )
        _LOGGER.info(f'Size of attn layer: {attn.size()}')
        # apply dropout
        output = self._dropout1(attn)
        _LOGGER.info(f'Size of dropout: {output.size()}')
        
        # resildual from raw inputs + layernorm
        output = self._norm2(output + input_context)
        _LOGGER.info(f'Size of residual: {output.size()}')

        # residual connection + mlp
        output = output + self._mlp(output)
        _LOGGER.info(f'Size of output + mlp: {output.size()}')
        
        return output

    def forward(self, x, key_padding_mask):
        # x is in the shape [sequence_length, batch_size, model_dim]
        return self.forward_pre(x)


class GPT(nn.Module):
    def __init__(self, 
                 n_vocab=1000, 
                 n_layers=8, 
                 model_dim=256,
                 dim_ffn=2048,
                 n_heads=8,
                 dropout=0.1,
                 max_seq_length=1024):
        super(GPT, self).__init__()
        self._model_dim = model_dim
        self._dim_ffn = dim_ffn
        self._n_heads= n_heads
        self._dropout = dropout
        self._n_vocab = n_vocab
        self._n_layers = n_layers
        self._max_seq_length = max_seq_length

        # initialize token embeddings and positional encodings 
        self._token_embeddings = nn.Embedding(n_vocab, model_dim)
        self._positional_encodings = nn.Embedding(max_seq_length, model_dim)
        
        self._attn_blocks = nn.Sequential(*[
            GPTDecoderLayer(
                model_dim=model_dim, 
                n_heads=n_heads, 
                dim_ffn=dim_ffn, 
                dropout=dropout)
            for x in range(n_layers)
        ])

        self._norm1 = nn.LayerNorm(model_dim)
        self._head =  nn.Linear(model_dim, n_vocab)
        self._dropout1 = nn.Dropout(dropout)
        
    def forward(self, input_tokens, key_padding_mask=None):
        """
        Args:
            input_tokens: shape (batch_size, sequence_length)
            key_padding_mask: shape (batch_size, max_seq_length) mask to padded entries
            target_tokens: shape (batch_size, sequence_length) Optional for training
        """
        device = input_tokens.device
        seq_len = input_tokens.size(1)
        token_embed = self._token_embeddings(input_tokens)
        positions = torch.arange(0, seq_len, device=device)
        pos_encoding = self._positional_encodings(positions)

        out = self._dropout1(token_embed + pos_encoding)
        _LOGGER.info(f'Size of embeddings+pos_encoding {out.size()}')
    
        for i in range(self._n_layers):
            out = self._attn_blocks[i](out, key_padding_mask)
            _LOGGER.info(f'Size of attention layer {i} -> {out.size()}')

        out = self._norm1(out)
        logits = self._head(out)

        _LOGGER.info(f'Size of logits {logits.size()}')


        return logits



### Training a GPT architecture

For a dataset of input and target sequence pairs ${(x_i, y_i)}_{i=1}^N$, we want to estimate the parameters of the model $\theta$ that maximizes the likelihood:

$$
\mathcal{L} = \prod_{i=1}^N P_{\theta}(x_i)
$$

and applying the log is:

$$
\mathcal{l} = \sum_{i=1}^N \log P_{\theta}(x_i).
$$

Now replacing the probability for sequence $x_i$:

$$
\mathcal{l} = \sum_{i=1}^N \log \left(  \prod_{t=1}^T P_{\theta}(x_{i,t} | x_{i, <t}) \right),
$$

$$
\mathcal{l} = \sum_{i=1}^N \sum_{t=1}^T \log P_{\theta}(x_{i,t} | x_{i, <t}).
$$


Now, for a K-class probability distribution we write $P(x=c) = \prod_{k=1}^K P(k)^{I[k=c]}$ and apply the log properties to obtain

$$
\mathcal{l} = \sum_{i=1}^N \sum_{t=1}^T \sum_{k=1}^K I[k=c] \log P_{\theta}(x_{i,t}=c | x_{i, <t}),
$$

where $I$ is an indicator function that is $1$ when the inner expresion is true. By knowing the true target sequence the indicator fuction will zero-out the terms in the outer sum different to the true value, then we can eliminate the rest of $K$ elements in the sum


$$
\mathcal{l} = \sum_{i=1}^N \sum_{t=1}^T \log P_{\theta}(x_{i,t}=y_{i,t} | x_{i, <t}).
$$


In [ ]:

from torch.utils.data import DataLoader
# from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim
import os


MAX_SEQ_LEN = 1024
N_HEADS = 8
FFN_DIM = 2048
N_DIM = 256
N_VOCAB = tokenizer.vocab_size
N_LAYERS = 8
DROPOUT = 0.1
BATCH_SIZE = 16
LEARNING_RATE = 3e-4

project_path = '/home/ubuntu/efs/gonamang/src/glamroom/animation/notebooks/gpt_project'
model_path  = f'{project_path}/models'

os.makedirs(model_path, exist_ok=True)

class GPTInterface():
    def __init__(self,
            n_vocab=N_VOCAB, 
            n_layers=N_LAYERS, 
            model_dim=FFN_DIM,
            dim_ffn=FFN_DIM,
            n_heads=N_HEADS,
            dropout=DROPOUT,
            max_seq_length=MAX_SEQ_LEN):
        
        self._gpt_model = GPT(
            n_vocab=n_vocab, 
            n_layers=n_layers, 
            model_dim=model_dim,
            dim_ffn=dim_ffn,
            n_heads=n_heads,
            dropout=dropout,
            max_seq_length=max_seq_length
        )

        self._optimizer = optim.AdamW(
            self._gpt_model.parameters(), 
            lr=LEARNING_RATE,
            betas=(0.9, 0.999),
            weight_decay=0.1
        )
        self._total_batches = 0
        self._write_every_batches = 1000
        self._save_every_batches = 10000

        #self._writer = SummaryWriter(
        #    os.path.join(project_path, 'tf_logs')
        #)
        

    def train_one_epoch(self, train_loader, device):
        self._gpt_model.train()
        N = len(train_dataset)

        loss_ce = nn.CrossEntropyLoss(reduction='none')
    
        epoch_loss = 0
        for n, batch in enumerate(tqdm.tqdm(train_loader, total=N)):
            self._optimizer.zero_grad()
            print(batch.keys())
            input_tokens = batch['input_ids'].to(device)
            inputs = input_tokens[:, :-1]
            targets = input_tokens[:, 1:]
            key_mask_padding = batch['attention_mask'].to(device)[:, 1:]

            # batch size
            B = targets.size(0)
            
            outputs = self._gpt_model(inputs, key_mask_padding)
            _LOGGER.info(f'Size of preds: {outputs.size()} and targets: {targets.size()}')
            # [B*seq_len, n_vocab]
            outputs = outputs.view(-1, self._gpt_model._n_vocab)
            # [B*seq_len]
            
            labels = targets.reshape(-1)
    
            loss = loss_ce(outputs, labels)
            # [B, seq_len]
            loss = loss.view(B, -1)
    
            # apply mask then sum
            loss = (loss * key_mask_padding).sum()
            loss.backward()
            self._optimizer.step()
    
            if self._total_batches % self._write_every_batches:
                self.write_scalars('loss/batch_loss', {'train': loss.item()}, self._total_batches)
    
            self._total_batches += 1
            epoch_loss += loss.item()
    
        metrics_dict['epoch_loss'] = epoch_loss / N
        return metrics_dict

    def write_scalars(self, metric_tab, metric_dict, time):
        #self._writer.add_scalars(
        #    metric_str, 
        #    metric_dict, 
        #    time
        #)
        pass

    def fit(self, train_loader, device, n_epochs=10):
        self._total_batches = 0
        self._gpt_model.to(device)

        for e in range(n_epochs):
            metrics_dict = self.train_one_epoch(train_loader, device)

            if self._total_batches % self._save_every_batches == 0:
                torch.save(self._gpt_model.state_dict(), model_path)


def preprocess_dataset(input_dataset):
    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding="max_length",
            return_attention_mask=True,
        )
    
    tokenized_dataset = input_dataset.map(
        tokenize_batch,
        batched=True,
        batch_size=1000,
        num_proc=4,
        remove_columns=input_dataset.column_names,
        desc='Preparing data for Pytorch loader'
    )
    
    tokenized_dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask"],
    )

    return tokenized_dataset



In [ ]:

tokenizer.pad_token = tokenizer.eos_token
train_dataset_ready = preprocess_dataset(train_dataset_final)
train_loader = DataLoader(
    train_dataset_ready,
    batch_size=BATCH_SIZE,
    shuffle=True,
)



In [ ]:
print(N_VOCAB)

In [ ]:

gpt = GPTInterface()
device = torch.device('cuda:0')
gpt.fit(train_loader, device)



In [ ]:


# train_dataset_unique_ppl.save_to_disk("tinystories_train_ppl")
# val_dataset_unique_ppl.save_to_disk("tinystories_val_ppl")
# train_dataset_ready.save_to_disk("tinystories_train_dataset_ready")


In [ ]:
print('Hola')

In [ ]:
# Clear PyTorch CUDA cache
torch.cuda.empty_cache()

# Optional: collect IPC memory
torch.cuda.ipc_collect()

In [ ]:
import gc
# Run Python garbage collector
gc.collect()

In [ ]:
import math
import matplotlib.pyplot as plt

total_steps = 10*95532
warmup_steps=5000
num_cycles=0.5
min_lr=1e-8

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step)/float(warmup_steps)
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(min_lr, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress)))


x_list = list(range(0, total_steps))
y_list = [lr_lambda(x) for x in x_list]

limit = -1

plt.plot(x_list[:limit], y_list[:limit])


